In [1]:
import warnings
warnings.filterwarnings('ignore')

(Tutorial_Get_tpsa)=
# Get TPSA

*Calculating topological polar surface areas for complete molecular graphs.*

The experimental {func}`molsysmt.physchem.get_tpsa` implements the Ertl,
Rohde and Selzer model through RDKit. It returns one area per selected molecule.
TPSA is a graph descriptor, not geometric SASA or a permeability prediction.
No conformer generation, protonation preparation or source mutation occurs.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

See {func}`molsysmt.physchem.get_tpsa` for input requirements and diagnostics.
:::


## Basic usage

Let's show how this function works with two fully defined chemical graphs.
RDKit is an optional explicit provider. Inputs may use any supported form with
the required chemistry and molecule membership. No coordinates are needed.

In [2]:
import molsysmt as msm
from rdkit import Chem

molsys = msm.convert(Chem.MolFromSmiles('CCO.COC'), to_form='molsysmt.MolSys')
area = msm.physchem.get_tpsa(molsys)
msm.pyunitwizard.get_value(area, to_unit='angstrom**2').round(2).tolist()

[20.23, 9.23]

## Selecting whole molecules

Numeric selections are molecule indices and retain order and repetitions.
String expressions use `syntax`; an expression identifying an atom selects its
whole parent molecule rather than cutting a chemically different fragment.
The default `element` is `molecule`. Other element levels return model
contributions in complete molecular context, as shown below. Separate molecules
retain separate default values.

In [3]:
repeated = msm.physchem.get_tpsa(molsys, selection=[1, 0, 1], return_report=True)
selected = msm.physchem.get_tpsa(molsys, selection='atom_index==0')
repeated['report']['molecule_indices'].tolist(), repeated['report']['atom_offsets'].tolist(), selected.shape

([1, 0, 1], [0, 3, 6, 9], (1,))

## Choosing the polar elements

The default counts N/O contributions, following the conventional reference
program/RDKit variant. Set `include_sulfur_phosphorus=True` to include polar
S/P contributions from the published fragment table. Keep this choice explicit
when comparing values or building a dataset. Neither variant is universally better.

In [4]:
molsys = msm.convert(Chem.MolFromSmiles('OC(=O)P(=O)(O)O'), to_form='molsysmt.MolSys')
{
    enabled: msm.pyunitwizard.get_value(
        msm.physchem.get_tpsa(molsys, include_sulfur_phosphorus=enabled),
        to_unit='angstrom**2',
    ).round(2).tolist()
    for enabled in (False, True)
}

{False: [94.83], True: [104.64]}

## Using the active unit policy

The area is a PyUnitWizard quantity: 20.23 angstrom squared corresponds to
0.2023 nm squared. Unlike bare descriptor numbers, its value and unit remain
coupled when the application changes its standard length unit.

In [5]:
molsys = msm.convert(Chem.MolFromSmiles('CCO'), to_form='molsysmt.Topology')
with msm.pyunitwizard.context(standard_units=['pm', 'ps', 'radians']):
    area = msm.physchem.get_tpsa(molsys)
    empty = msm.physchem.get_tpsa(molsys, selection=[])
    print(msm.pyunitwizard.get_unit(area))
    print(msm.pyunitwizard.get_value(area, to_unit='nm**2').round(4).tolist())
    print(empty.shape)

picometer ** 2
[0.2023]
(0,)


## Declaring the chemical state

`chemical_state='reference'` uses the declared reference, not the state of the
first selected structure. An integer chooses a stored state. A `structure`
request must resolve unambiguously across `structure_indices`. This graph-based
property has no structures output axis. Spatial selection may require coordinates.

Charged molecular graphs are allowed, but charges, bond orders and the hydrogen
inventory must already be defined. Indexed H and declared nonindexed H are
supported; the function never prepares a missing protonation state. Native SDF
inputs need explicit valence/H evidence. A default CTAB valence does not supply
that evidence; such inputs fail clearly rather than receiving inferred counts.

In [6]:
molsys = msm.convert(Chem.MolFromSmiles('CC(=O)[O-]'), to_form='molsysmt.MolSys')
result = msm.physchem.get_tpsa(molsys, chemical_state='reference', return_report=True)
result['report']['formal_charges'].tolist(), msm.pyunitwizard.get_value(result['tpsa'], to_unit='angstrom**2').round(2).tolist()

([-1], [40.13])

## Retaining the provenance

`return_report=True` returns `tpsa` and a detached `report`. Source molecule
indices and flattened atom membership with offsets identify the evaluated
population. The report records the state/digest, method, element scope,
hydrogen policy, original software versions and bibliography. The quantity
retains its unit; the metadata report alone does not contain area values.
No automatic H5MSM attachment or standalone JSON serialization is implied.
Optional Ackredit contributes to the application's session and preserves
completed results if attribution is unavailable or fails.

In [7]:
report = result['report']
print(report['method'], report['parameters'])
print(report['atom_indices'].tolist(), report['atom_offsets'].tolist())
print([item['id'] for item in report['attribution']['items']])

ertl_rohde_selzer {'include_sulfur_phosphorus': False, 'hydrogen_treatment': 'preserve_declared_inventory', 'implementation': 'rdkit.CalcTPSA'}
[0, 1, 2, 3] [0, 4]
['rdkit-aromaticity-model', 'software:molsysmt:0.23.0+201.g3502232d8.dirty', 'software:rdkit:2025.09.5', 'doi:10.1021/jm000942e']


## Selecting atomic contributions

Use `element='atom'` to select atom indices. Contributions are evaluated on the
complete parent molecule, including atoms outside the output selection. They
are not descriptors of isolated capped fragments. Numeric order and repetitions
are retained, and each needed parent molecule is evaluated once.


In [8]:
molsys = msm.convert(Chem.MolFromSmiles('CCO'), to_form='molsysmt.MolSys')
atomic = msm.physchem.get_tpsa(molsys, element='atom', selection=[2, 0, 2])
msm.pyunitwizard.get_value(atomic, to_unit='angstrom**2').round(4).tolist()


[20.23, 0.0, 20.23]

## Grouping the contributions

The same pattern accepts `group`, `component`, `chain` and `entity`. Numeric
selection addresses that element level; text selects whole elements. Here we
explicitly define two groups in one molecule to show that grouping does not
change the chemical graph or require cutting its bonds. The polar group contains
atoms 1 and 2, and the alkyl group contains atom 0. These are illustrative groups,
not a rule for partitioning ligands.


In [9]:
molsys.topology.reset_groups(n_groups=2)
molsys.topology.groups['group_name'] = ['POLAR', 'ALKYL']
molsys.topology.groups['group_type'] = ['small molecule', 'small molecule']
molsys.topology.groups['molecule_index'] = [0, 0]
molsys.topology.atoms['group_index'] = [1, 0, 0]
grouped = msm.physchem.get_tpsa(molsys, element='group', selection=[1, 0, 1])
msm.pyunitwizard.get_value(grouped, to_unit='angstrom**2').round(4).tolist()


[0.0, 20.23, 0.0]

## Summing a selection

Use `element='system'` for a scalar sum over selected atoms. Numeric selections
are atom indices in this mode. Repeated indices repeat their contribution; an
empty atom selection gives zero. At other element levels an empty selection
returns shape `(0,)`. Sums spanning molecules are additive model contributions,
not a physical mixture partition coefficient or exposed surface measurement.


In [10]:
total = msm.physchem.get_tpsa(molsys, element='system', selection=[0, 2])
float(msm.pyunitwizard.get_value(total, to_unit='angstrom**2'))


20.230000000000004

## Interpreting hydrogen attribution and source maps

A hydrogen with a source atom index keeps its own contribution. A nonindexed
hydrogen contributes to its retained attachment atom. Therefore individual heavy
atom contributions can differ between explicit and implicit H representations;
folding indexed H contributions onto their attachment atoms restores agreement.
TPSA assigns zero to H atoms; their chemical context still determines polar-atom
contributions. No source atoms are added or removed by either calculation.

In a report, `element_indices`, `element_atom_indices` and
`element_atom_offsets` identify output rows. `molecule_indices`, `atom_indices`
and `atom_offsets` identify the complete molecules evaluated as context. Their
row counts can differ. `output_kind` identifies contribution sums versus complete
molecular descriptors. Charge and charged-site arrays refer to these evaluated
molecular contexts. Keep the distinction when exporting a subset.


In [11]:
result = msm.physchem.get_tpsa(molsys, element='atom', selection=[2, 0, 2], return_report=True)
report = result['report']
print(report['element'], report['element_indices'].tolist())
print(report['element_atom_indices'].tolist(), report['element_atom_offsets'].tolist())
print(report['molecule_indices'].tolist(), report['atom_indices'].tolist())
print(report['output_kind'])


atom [2, 0, 2]
[2, 0, 2] [0, 1, 2, 3]
[0] [0, 1, 2]
atomic_contribution_sum


## Understanding the limits

A selected molecule must be covalently complete and connected. A label containing
several salt components, or a covalent bond crossing a molecule boundary, fails.
The initial route excludes metals, dative bonds within a molecule, radicals and
query/dummy atoms. Chemical contradictions and missing evidence fail explicitly.
Geometry, solvent shielding and environmental pKa are outside this model.

The method reference is [Ertl, Rohde and Selzer (2000)](https://doi.org/10.1021/jm000942e).
The [RDKit Book](https://www.rdkit.org/docs/RDKit_Book.html#implementation-of-the-tpsa-descriptor)
explains the two polar-element variants. Original-model attribution is distinct
from the executed RDKit implementation.

:::{seealso}
:class: dropdown

- {func}`molsysmt.basic.convert`: supplying an appropriate molecular form.
- {func}`molsysmt.physchem.get_tpsa`: calculating the graph descriptor.
- {func}`molsysmt.physchem.get_sasa`: calculating geometric solvent exposure.
- {ref}`Cookbook_TPSA_Descriptors`: exporting indexed molecular descriptors.
:::
